<a href="https://colab.research.google.com/github/muskan-rathor/Semantic-Communication/blob/main/semantic_communication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import requests
import zipfile
import os
from PIL import Image
from torch.utils.data import random_split
import random
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
from itertools import cycle
import torch.nn.functional as F

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## --- 1. Task-Specific Label Mapping ---
# We map the 43 GTSRB classes to our specific task labels.

# Shape labels: 0:Circle, 1:Triangle, 2:Octagon, 3:Square/Rect
SHAPE_MAP = {
    0:0, 1:0, 2:0, 3:0, 4:0, 5:0, 6:0, 7:0, 8:0, 9:0, 10:0, 14:2, 15:0, 16:0,
    17:0, 18:1, 19:1, 20:1, 21:1, 22:1, 23:1, 24:1, 25:1, 26:1, 27:1, 28:1,
    29:1, 30:1, 31:1, 32:0, 33:3, 34:3, 35:3, 36:3, 37:3, 38:3, 39:3, 40:3,
    41:0, 42:0, 11:1, 12:3, 13:1
}
SHAPE_CLASSES = {0:'Circle', 1:'Triangle', 2:'Octagon', 3:'Square/Rect'}

# Color labels: 0:Red, 1:Blue, 2:Black/White, 3:Yellow
COLOR_MAP = {
    0:0, 1:0, 2:0, 3:0, 4:0, 5:0, 7:0, 8:0, 9:0, 10:0, 14:0, 15:2, 16:0,
    17:0, 18:0, 19:2, 20:2, 21:2, 22:2, 23:2, 24:2, 25:2, 26:0, 27:2,
    28:2, 29:2, 30:2, 31:2, 32:2, 33:1, 34:1, 35:1, 36:1, 37:1, 38:1,
    39:1, 40:1, 41:2, 42:2, 6:2, 11:0, 12:0, 13:0
}
COLOR_CLASSES = {0:'Red', 1:'Blue', 2:'Black/White'} # No yellow signs in this mapping

# Speed Limit labels: 0:No, 1:Yes
SPEED_MAP = {
    0:1, 1:1, 2:1, 3:1, 4:1, 5:1, 6:0, 7:1, 8:1, 32:0, 41:0, 42:0
}
SPEED_CLASSES = {0:'Not Speed Limit', 1:'Speed Limit'}


class GTSRBMultiTaskDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = os.path.join(data_dir, "GTSRB", "Final_Training", "Images")
        self.transform = transform
        self.images = []
        self.original_labels = []
        if not os.path.exists(self.data_dir):
            raise FileNotFoundError(f"Dataset directory not found at {self.data_dir}")
        print("Loading dataset paths...")
        for class_dir in sorted(os.listdir(self.data_dir)):
            class_path = os.path.join(self.data_dir, class_dir)
            if os.path.isdir(class_path):
                class_id = int(class_dir)
                for img_name in os.listdir(class_path):
                    if img_name.endswith('.ppm'):
                        self.images.append(os.path.join(class_path, img_name))
                        self.original_labels.append(class_id)
        print(f"Loaded {len(self.images)} image paths.")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        original_label = self.original_labels[idx]
        image = Image.open(img_path).convert('RGB')

        # We need the original image for visualization later
        if self.transform:
            transformed_image = self.transform(image)

        # Map original label to task-specific labels
        labels = {
            'shape': SHAPE_MAP.get(original_label, -1),
            'color': COLOR_MAP.get(original_label, -1),
            'speed': SPEED_MAP.get(original_label, 0) # Default to 'No'
        }
        return transformed_image, labels

# --- Data Preparation ---
def download_and_extract_dataset(url, extract_to="data"):
    if not os.path.exists(extract_to):
        os.makedirs(extract_to)
        zip_path = "GTSRB_Final_Training_Images.zip"
        print("Downloading dataset...")
        response = requests.get(url, stream=True)
        with open(zip_path, "wb") as f:
            f.write(response.content)
        print("Dataset downloaded. Extracting...")
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(extract_to)
        print("Dataset extracted.")
        os.remove(zip_path)
    else:
        print("Dataset already exists.")
    return extract_to


# --- Model Architecture ---
class CNNEncoder(nn.Module):
    def __init__(self, latent_dim):
        super(CNNEncoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1), nn.ReLU(True),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1), nn.ReLU(True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1), nn.ReLU(True),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1), nn.ReLU(True),
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 1024), nn.ReLU(True),
            nn.Linear(1024, latent_dim)
        )
    def forward(self, x):
        return self.encoder(x)

## --- 2. Multi-Headed Decoder Architecture ---
class TaskDecoder(nn.Module):
    """A generic decoder for a single task."""
    def __init__(self, latent_dim, num_classes):
        super(TaskDecoder, self).__init__()
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.ReLU(True),
            nn.Linear(256, 128), nn.ReLU(True),
            nn.Linear(128, num_classes)
        )
    def forward(self, z):
        return self.decoder(z)

class AttentionReasoningModule(nn.Module):
    def __init__(self, latent_dim, num_tasks, num_heads=4):
        super().__init__()
        self.latent_dim = latent_dim
        self.num_tasks = num_tasks

        self.attention = nn.MultiheadAttention(
            embed_dim=latent_dim,
            num_heads=num_heads,
            batch_first=True
        )
        self.task_queries = nn.Parameter(
            torch.randn(num_tasks, latent_dim)
        )
        self.feature_gate = nn.Sequential(
            nn.Linear(latent_dim, latent_dim // 2),
            nn.ReLU(),
            nn.Linear(latent_dim // 2, latent_dim),
            nn.Sigmoid()
        )

    def forward(self, z, task_id, message_length):
        batch_size = z.shape[0]
        query = self.task_queries[task_id].unsqueeze(0).repeat(batch_size, 1, 1)
        z_expanded = z.unsqueeze(1)
        attn_output, _ = self.attention(query, z_expanded, z_expanded)
        importance_scores = self.feature_gate(attn_output.squeeze(1))
        _, top_indices = torch.topk(importance_scores, message_length, dim=1)
        z_R = torch.zeros_like(z)
        for i in range(batch_size):
            z_R[i, top_indices[i]] = z[i, top_indices[i]]
        return z_R, importance_scores

# --- Channel Simulation ---
def awgn_channel(z, snr_db):
    snr_linear = 10 ** (snr_db / 10.0)
    power = torch.mean(z ** 2, dim=1, keepdim=True)
    noise_power = power / (snr_linear + 1e-9)
    noise = torch.randn_like(z) * torch.sqrt(noise_power)
    return z + noise

## --- New plotting functions ---
def plot_training_loss(losses):
    """Plots the training loss over steps."""
    plt.rcParams.update({'font.size': 20, 'font.weight': 'bold', 'axes.labelweight': 'bold'})
    plt.figure(figsize=(10, 6))
    plt.plot(losses)
    # plt.title("Training Loss Curve", fontsize=20)
    plt.xlabel("Training Step (x100)", fontsize=20)
    plt.ylabel("Total Loss", fontsize=20)
    plt.grid(True)
    plt.savefig("training_loss_curve.eps", format='eps', dpi=700)
    plt.show()
    print("Plot saved as training_loss_curve.eps")

def plot_confusion_matrices(viz_data, plot_suffix):
    """Plots a confusion matrix for each task."""
    plt.rcParams.update({'font.size': 20, 'font.weight': 'bold', 'axes.labelweight': 'bold'})
    for task_name, data in viz_data.items():
        cm = confusion_matrix(data['y_true'], data['y_pred'])
        class_names = list(data['classes'].values())

        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=class_names, yticklabels=class_names,
                    annot_kws={"weight": "bold", "size": 18}) # Adjusted annotation size for clarity
        # plt.title(f"Confusion Matrix for {task_name} ({plot_suffix})", fontsize=20)
        plt.xlabel("Predicted Label", fontsize=20)
        plt.ylabel("True Label", fontsize=20)
        filename = f"confusion_matrix_{task_name.lower().replace(' ', '_')}.eps"
        plt.savefig(filename, format='eps', dpi=700)
        plt.show()
        print(f"Plot saved as {filename}")

def plot_roc_curves(viz_data, plot_suffix):
    """Plots ROC curves for each class of each task."""
    plt.rcParams.update({'font.size': 20, 'font.weight': 'bold', 'axes.labelweight': 'bold'})
    for task_name, data in viz_data.items():
        y_true = np.array(data['y_true'])
        y_score = np.array(data['y_score'])
        class_map = data['classes']
        n_classes = len(class_map)

        if n_classes <= 2: # Binary case
             fpr, tpr, _ = roc_curve(y_true, y_score[:, 1])
             roc_auc = auc(fpr, tpr)
             plt.figure(figsize=(8,6))
             plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
             plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
        else: # Multi-class case
            y_true_bin = label_binarize(y_true, classes=list(class_map.keys()))
            fpr = dict()
            tpr = dict()
            roc_auc = dict()
            for i in range(n_classes):
                fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_score[:, i])
                roc_auc[i] = auc(fpr[i], tpr[i])

            plt.figure(figsize=(10, 8))
            colors = cycle(['aqua', 'darkorange', 'cornflowerblue', 'deeppink', 'navy', 'green'])
            for i, color in zip(range(n_classes), colors):
                plt.plot(fpr[i], tpr[i], color=color, lw=2,
                         label=f'ROC curve of class {class_map[i]} (area = {roc_auc[i]:.2f})')
            plt.plot([0, 1], [0, 1], 'k--', lw=2)

        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate', fontsize=20)
        plt.ylabel('True Positive Rate', fontsize=20)
        # plt.title(f'ROC Curve for {task_name} ({plot_suffix})', fontsize=20)
        plt.legend(loc="lower right", fontsize=16) # Adjusted legend size for clarity
        filename = f"roc_curve_{task_name.lower().replace(' ', '_')}.eps"
        plt.savefig(filename, format='eps', dpi=700)
        plt.show()
        print(f"Plot saved as {filename}")

def visualize_attention_scores(dataset, encoder, attention_module, device, num_samples=3):
    """Visualizes attention feature importance for a few random samples."""
    plt.rcParams.update({'font.size': 20, 'font.weight': 'bold', 'axes.labelweight': 'bold'})
    # Temporarily create a dataset that doesn't transform to tensor
    vis_transform = transforms.Compose([transforms.Resize((64, 64))])
    raw_dataset = GTSRBMultiTaskDataset(dataset.dataset.data_dir.split('/GTSRB')[0], transform=vis_transform)

    indices = random.sample(range(len(raw_dataset)), num_samples)

    # Prepare images for model and for display
    tensor_images = []
    display_images = []
    for idx in indices:
        img_pil, _ = raw_dataset[idx]
        display_images.append(img_pil)
        # Apply the full transform for model input
        tensor_images.append(dataset.dataset.transform(img_pil))

    images_tensor = torch.stack(tensor_images).to(device)

    encoder.eval()
    attention_module.eval()
    with torch.no_grad():
        z = encoder(images_tensor)
        # Arbitrarily using a message length of half the latent dim for visualization
        message_length = z.shape[1] // 2
        _, scores_shape = attention_module(z, 0, message_length)
        _, scores_color = attention_module(z, 1, message_length)
        _, scores_speed = attention_module(z, 2, message_length)

    scores_shape = scores_shape.cpu().numpy()
    scores_color = scores_color.cpu().numpy()
    scores_speed = scores_speed.cpu().numpy()

    fig, axes = plt.subplots(num_samples, 4, figsize=(20, 5 * num_samples))
    # fig.suptitle("Attention-Based Feature Importance Scores for Different Tasks", fontsize=22)

    for i in range(num_samples):
        # Display original image
        axes[i, 0].imshow(display_images[i])
        # axes[i, 0].set_title(f"Sample Image {i+1}", fontsize=20)
        axes[i, 0].axis('off')

        # Plot scores for shape
        axes[i, 1].bar(range(z.shape[1]), scores_shape[i])
        # axes[i, 1].set_title("Task: Shape", fontsize=20)
        axes[i, 1].set_ylabel("Importance", fontsize=20)

        # Plot scores for color
        axes[i, 2].bar(range(z.shape[1]), scores_color[i])
        # axes[i, 2].set_title("Task: Color", fontsize=20)

        # Plot scores for speed
        axes[i, 3].bar(range(z.shape[1]), scores_speed[i])
        # axes[i, 3].set_title("Task: Speed Limit", fontsize=20)

    for ax in axes[:, 1:].flatten():
        ax.set_ylim(0, 1)
        ax.set_xlabel("Latent Feature Index", fontsize=20)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig("attention_scores_visualization.eps", format='eps', dpi=700)
    plt.show()
    print("Plot saved as attention_scores_visualization.eps")

# --- Main Pipeline ---
def train_and_evaluate_multitask():
    # Hyperparameters
    latent_dim = 128
    num_epochs = 15
    batch_size = 128
    learning_rate = 1e-4
    num_tasks = 3

    # Data Preparation
    data_dir = download_and_extract_dataset("https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Training_Images.zip")
    transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.3403, 0.3121, 0.3214], std=[0.2724, 0.2608, 0.2669])
    ])
    full_dataset = GTSRBMultiTaskDataset(data_dir, transform=transform)
    train_size = int(0.8 * len(full_dataset))
    test_size = len(full_dataset) - train_size
    train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    # Models and Optimizer
    encoder = CNNEncoder(latent_dim).to(device)
    shape_decoder = TaskDecoder(latent_dim, len(SHAPE_CLASSES)).to(device)
    color_decoder = TaskDecoder(latent_dim, len(COLOR_CLASSES)).to(device)
    speed_decoder = TaskDecoder(latent_dim, len(SPEED_CLASSES)).to(device)
    attention_module = AttentionReasoningModule(latent_dim, num_tasks).to(device)

    all_params = list(encoder.parameters()) + list(shape_decoder.parameters()) + \
                 list(color_decoder.parameters()) + list(speed_decoder.parameters()) + \
                 list(attention_module.parameters())
    optimizer = optim.Adam(all_params, lr=learning_rate)
    criterion = nn.CrossEntropyLoss()

    train_losses = [] # To store loss values for plotting

    ## --- 3. Multi-Task Training Loop ---
    print("Starting multi-task training with attention-based feature selection...")
    for epoch in range(num_epochs):
        encoder.train()
        shape_decoder.train()
        color_decoder.train()
        speed_decoder.train()
        attention_module.train()
        for i, (images, labels_dict) in enumerate(train_loader):
            images = images.to(device)
            shape_labels = labels_dict['shape'].to(device)
            color_labels = labels_dict['color'].to(device)
            speed_labels = labels_dict['speed'].to(device)
            z = encoder(images)
            message_length = random.randint(1, latent_dim)
            snr_db = random.uniform(-10, 20)
            z_R_shape, _ = attention_module(z, task_id=0, message_length=message_length)
            z_noisy_shape = awgn_channel(z_R_shape, snr_db)
            shape_outputs = shape_decoder(z_noisy_shape)
            z_R_color, _ = attention_module(z, task_id=1, message_length=message_length)
            z_noisy_color = awgn_channel(z_R_color, snr_db)
            color_outputs = color_decoder(z_noisy_color)
            z_R_speed, _ = attention_module(z, task_id=2, message_length=message_length)
            z_noisy_speed = awgn_channel(z_R_speed, snr_db)
            speed_outputs = speed_decoder(z_noisy_speed)
            loss_shape = criterion(shape_outputs, shape_labels)
            loss_color = criterion(color_outputs, color_labels)
            loss_speed = criterion(speed_outputs, speed_labels)
            total_loss = loss_shape + loss_color + loss_speed
            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()
            if (i + 1) % 100 == 0:
                print(f"Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Total Loss: {total_loss.item():.4f}")
                train_losses.append(total_loss.item())

    ## --- 4. Multi-Task Evaluation ---
    print("Starting multi-task evaluation...")
    encoder.eval()
    shape_decoder.eval()
    color_decoder.eval()
    speed_decoder.eval()
    attention_module.eval()

    tasks = {
        'Shape': {'decoder': shape_decoder, 'task_id': 0, 'accuracies': {}, 'classes': SHAPE_CLASSES},
        'Color': {'decoder': color_decoder, 'task_id': 1, 'accuracies': {}, 'classes': COLOR_CLASSES},
        'Speed Limit': {'decoder': speed_decoder, 'task_id': 2, 'accuracies': {}, 'classes': SPEED_CLASSES},
    }

    message_lengths = [1, 2, 3, 4]
    snr_dbs = np.linspace(-40, 20, 16)
    for length in message_lengths:
        for task_name in tasks:
            tasks[task_name]['accuracies'][length] = []

    with torch.no_grad():
        for snr in snr_dbs:
            correct_counters = {task: {length: 0 for length in message_lengths} for task in tasks}
            total_samples = 0
            for images, labels_dict in test_loader:
                images = images.to(device)
                total_samples += images.size(0)
                z = encoder(images)
                for length in message_lengths:
                    for task_name, task_info in tasks.items():
                        task_id = task_info['task_id']
                        decoder = task_info['decoder']
                        labels = labels_dict[task_name.lower().replace(' limit', '')].to(device)
                        z_R, _ = attention_module(z, task_id, length)
                        z_R_noisy = awgn_channel(z_R, snr)
                        outputs = decoder(z_R_noisy)
                        _, predicted = torch.max(outputs.data, 1)
                        correct_counters[task_name][length] += (predicted == labels).sum().item()
            for task_name in tasks:
                for length in message_lengths:
                    accuracy = 100 * correct_counters[task_name][length] / total_samples
                    tasks[task_name]['accuracies'][length].append(accuracy)
            print(f"SNR: {snr:.1f} dB | Accuracies computed for all tasks and lengths.")

    # --- 5. Plotting Results ---
    print("Plotting results for each task...")
    plt.rcParams.update({'font.size': 20, 'font.weight': 'bold', 'axes.labelweight': 'bold'})
    for task_name, task_info in tasks.items():
        plt.figure(figsize=(12, 8))
        for length in message_lengths:
            plt.plot(snr_dbs, task_info['accuracies'][length], marker='o', linestyle='-', label=f'Message Length = {length}')
        # plt.title(f"{task_name} Classification Accuracy vs. SNR", fontsize=20)
        plt.xlabel("SNR (dB)", fontsize=20)
        plt.ylabel("Accuracy (%)", fontsize=20)
        plt.legend(fontsize=18) # Adjusted legend size
        plt.grid(True)
        plt.ylim(0, 105)
        filename = f"accuracy_{task_name.lower().replace(' ', '_')}.eps"
        plt.savefig(filename, format='eps', dpi=700)
        plt.show()
        print(f"Plot saved as {filename}")

    ## --- 6. Additional Visualizations (ROC, Confusion Matrix, Attention) ---
    print("\n--- Generating Additional Visualizations ---")
    target_snr = -10.0
    target_length = 4

    # --- Data collection for ROC and Confusion Matrix ---
    print(f"Collecting data for plots at SNR={target_snr} dB, Message Length={target_length}...")
    viz_data = {
        task_name: {'y_true': [], 'y_score': [], 'y_pred': [], 'classes': info['classes']}
        for task_name, info in tasks.items()
    }

    with torch.no_grad():
        for images, labels_dict in test_loader:
            images = images.to(device)
            z = encoder(images)
            for task_name, task_info in tasks.items():
                decoder = task_info['decoder']
                task_id = task_info['task_id']
                key_name = task_name.lower().replace(' limit', '')
                labels = labels_dict[key_name].to(device)
                z_R, _ = attention_module(z, task_id, target_length)
                z_R_noisy = awgn_channel(z_R, target_snr)
                outputs = decoder(z_R_noisy)
                probs = F.softmax(outputs, dim=1)
                _, predicted = torch.max(outputs.data, 1)
                viz_data[task_name]['y_true'].extend(labels.cpu().numpy())
                viz_data[task_name]['y_score'].extend(probs.cpu().numpy())
                viz_data[task_name]['y_pred'].extend(predicted.cpu().numpy())

    # --- Plotting Training Loss ---
    plot_training_loss(train_losses)

    # --- Plotting ROC Curves and Confusion Matrices ---
    plot_suffix = f"SNR={target_snr}dB, L={target_length}"
    plot_roc_curves(viz_data, plot_suffix)
    plot_confusion_matrices(viz_data, plot_suffix)

    # --- Visualizing Attention Scores ---
    visualize_attention_scores(test_dataset, encoder, attention_module, device, num_samples=3)

if __name__ == "__main__":
    train_and_evaluate_multitask()